# Portfolio Optimiser

Optimises your holdings, explains **why** each position should change, and shows the charts.
This notebook uses the `portfolio_optimizer` package in this repository. For the interactive
dashboard run `streamlit run app.py` instead.

**Colab:** upload the repository (or `git clone` it), then run the first cell.

In [ ]:
# Run from the repository root (or a subfolder of it).
import os, sys
root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, root)
# !pip install -q -r {root}/requirements.txt   # uncomment on a fresh Colab runtime

In [ ]:
from portfolio_optimizer import Settings, analyze, demo_prices, fetch_prices
from portfolio_optimizer import charts
from portfolio_optimizer.textreport import render_text

portfolio = {
    "SILVERBETA.NS": {"qty": 2121, "avg_price": 198.64},
    "GOLDETF.NS": {"qty": 2950, "avg_price": 135.57},
    "HDFCSML250.NS": {"qty": 250, "avg_price": 173.13},
    "HAL.NS": {"qty": 500, "avg_price": 4278.3},
    "MAFANG.NS": {"qty": 100, "avg_price": 176.16},
}
BENCHMARK = "^NSEI"
USE_DEMO_DATA = False  # True = synthetic prices, no internet needed

settings = Settings(
    strategy="max_sharpe",    # "min_volatility", "risk_parity", "max_diversification"
    risk_free_rate=0.065,     # Indian 91-day T-bill ≈ 6.5%
    max_weight=0.40,          # no single holding above 40%
    rebalance_intensity=1.0,  # 0.5 = move halfway to the optimal weights
)

In [ ]:
tickers = list(portfolio)
load = demo_prices if USE_DEMO_DATA else fetch_prices
data = load(tickers + [BENCHMARK], start="2022-01-01")

report = analyze(portfolio, data[tickers], settings, data[BENCHMARK])
print(render_text(report, demo=USE_DEMO_DATA))

## Where the improvement comes from
Each holding's exact share of the Sharpe-ratio change. The bars add up to the total.

In [ ]:
charts.contribution_bars(report).show()

## Allocation and risk

In [ ]:
charts.allocation_bars(report).show()
charts.risk_vs_capital(report, "current").show()
charts.correlation_heatmap(report).show()

## Efficient frontier

In [ ]:
charts.frontier_chart(report).show()

## Technicals (timing only)

In [ ]:
for t in tickers:
    charts.technical_chart(report, t).update_layout(title=t).show()

## Walk-forward backtest (no hindsight)

In [ ]:
charts.backtest_chart(report).show()
report.backtest.stats.style.format({c: "{:.1%}" for c in ["cagr", "volatility", "max_drawdown", "var_95", "cvar_95"]}
                                   | {c: "{:.2f}" for c in ["sharpe", "sortino", "calmar", "beta"]})